In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
df = pd.read_csv("variant_summary.txt",sep="\t",low_memory=False)

In [ ]:
df.head(10)

In [ ]:
df.columns

In [ ]:
print(df.shape)
print(df.columns)

In [ ]:
df_snv = df[df['Type'] == 'single nucleotide variant']

In [ ]:
df_snv['ClinicalSignificance'].unique()

In [ ]:
valid_labels = [
    "Benign",
    "Likely benign",
    "Pathogenic",
    "Likely pathogenic",
    "Pathogenic/Likely pathogenic","Benign/Likely benign"
]
df_clean = df_snv[df_snv["ClinicalSignificance"].isin(valid_labels)]

In [ ]:
df_clean.head()

In [ ]:
df_clean.columns


In [ ]:
keep_columns =[
    "Type",
    "Name",
    "GeneSymbol",
    "ClinicalSignificance",
    "ReferenceAllele",
    "AlternateAllele",
    "Chromosome",
    "Start"
]
df_path_clean = df_clean[keep_columns]
df_path_clean.head()

In [ ]:
df_path_clean[["RefAA", "PositionAA", "AltAA"]] = df_path_clean["Name"].str.extract(
    r"\(p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})\)"
)

In [ ]:
df_path_clean

In [ ]:
df_path_clean["ReferenceAllele"].unique()

In [ ]:
df_path_clean = df_path_clean.dropna(subset=["RefAA","PositionAA","AltAA"])
df_path_clean = df_path_clean.drop_duplicates()
df_path_clean.head()

In [ ]:
df_path_clean.drop(columns=["ReferenceAllele","AlternateAllele"], inplace=True)

In [ ]:
df_path_clean.head()

In [ ]:
df = df_path_clean.drop_duplicates(subset=["GeneSymbol","RefAA","PositionAA","AltAA"])

In [ ]:
df.head(10)

In [ ]:
df.shape

In [ ]:
df.info()
df.shape

In [ ]:
df.drop(columns=["Type"], inplace=True)


In [ ]:
label_mapping  = {'Likely pathogenic':1, 'Pathogenic':1, 'Pathogenic/Likely pathogenic':1,
       'Benign':0, 'Benign/Likely benign':0, 'Likely benign':0}

In [ ]:

df["Label"] = df["ClinicalSignificance"].map(label_mapping)

df.head()

In [ ]:
df.drop(columns=['ClinicalSignificance'], inplace =True)

In [ ]:
df.head()

In [ ]:
sns.countplot(data=df, x="Label")
plt.title("Distribution of clinical significance classes")
plt.xlabel("Clinical significance")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()

In [ ]:
sns.histplot(data=df, x="PositionAA", bins=50, kde=True)
plt.title("Distribution of amino acid positions")
plt.xlabel("Amino acid position")
plt.ylabel("Count")
plt.show()

In [ ]:
df[df['Label']==1].count()

In [ ]:
df[df['Label']==0].count()


In [ ]:
top_genes = df_path_clean["GeneSymbol"].value_counts().head(15)
sns.barplot(x=top_genes.values, y=top_genes.index)
plt.title("Top 15 most frequent genes")
plt.xlabel("Count")
plt.ylabel("GeneSymbol")
plt.show()

In [ ]:
mutation_table = pd.crosstab(df["RefAA"], df["AltAA"])
plt.figure(figsize=(10, 8))
sns.heatmap(mutation_table, cmap="Blues")
plt.title("Amino acid substitution patterns")
plt.xlabel("Mutated amino acid")
plt.ylabel("Reference amino acid")
plt.show()

In [ ]:
aa_properties = {
'Ala': {'hydrophobic':1,'charge':0,'polar':0},
'Arg': {'hydrophobic':0,'charge':1,'polar':1},
'Asn': {'hydrophobic':0,'charge':0,'polar':1},
'Asp': {'hydrophobic':0,'charge':-1,'polar':1},
'Cys': {'hydrophobic':1,'charge':0,'polar':0},
'Gln': {'hydrophobic':0,'charge':0,'polar':1},
'Glu': {'hydrophobic':0,'charge':-1,'polar':1},
'Gly': {'hydrophobic':1,'charge':0,'polar':0},
'His': {'hydrophobic':0,'charge':1,'polar':1},
'Ile': {'hydrophobic':1,'charge':0,'polar':0},
'Leu': {'hydrophobic':1,'charge':0,'polar':0},
'Lys': {'hydrophobic':0,'charge':1,'polar':1},
'Met': {'hydrophobic':1,'charge':0,'polar':0},
'Phe': {'hydrophobic':1,'charge':0,'polar':0},
'Pro': {'hydrophobic':1,'charge':0,'polar':0},
'Ser': {'hydrophobic':0,'charge':0,'polar':1},
'Thr': {'hydrophobic':0,'charge':0,'polar':1},
'Trp': {'hydrophobic':1,'charge':0,'polar':0},
'Tyr': {'hydrophobic':0,'charge':0,'polar':1},
'Val': {'hydrophobic':1,'charge':0,'polar':0}
}

In [ ]:
df = df[
    (df_path_clean["RefAA"] != "Ter") &
    (df_path_clean["AltAA"] != "Ter")
]
df = df[
    (df_path_clean["RefAA"] != "Xaa") &
    (df_path_clean["AltAA"] != "Xaa")
]

In [ ]:
df.head()

In [ ]:
df["Ref_charge"] = df["RefAA"].map(lambda x: aa_properties[x]["charge"])
df["Alt_charge"] = df["AltAA"].map(lambda x: aa_properties[x]["charge"])
df["Ref_hydrophobic"] = df["RefAA"].map(lambda x: aa_properties[x]["hydrophobic"])
df["Alt_hydrophobic"] = df["AltAA"].map(lambda x: aa_properties[x]["hydrophobic"])
df["Ref_polarity"] = df["RefAA"].map(lambda x: aa_properties[x]["polar"])
df["Alt_polarity"] = df["AltAA"].map(lambda x: aa_properties[x]["polar"])

In [ ]:
df.head()

In [ ]:
df["charge_change"] = df["Alt_charge"] - df["Ref_charge"]
df["hydrophobic_change"] = df["Alt_hydrophobic"] - df["Ref_hydrophobic"]

In [ ]:
df.columns

In [ ]:
df_tt = df.drop(columns=["Name"])



In [ ]:
df_tt.head()

In [ ]:
df.head()

In [ ]:
X = df_tt.drop('Label', axis =1)

In [ ]:
X

In [ ]:
Y = df_tt['Label']

In [ ]:
Y

In [ ]:
Y.value_counts(normalize=True)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42,
    stratify=Y
)

In [ ]:
X

In [ ]:
X.info()

In [ ]:
categorical_cols = [
"GeneSymbol",
"Chromosome",
"RefAA",
"PositionAA",
"AltAA"
]

numeric_cols = [
"Start",
"Ref_charge",
"Alt_charge",
"Ref_hydrophobic",
"Alt_hydrophobic",
"Ref_polarity",
"Alt_polarity",
"charge_change",
"hydrophobic_change"
]

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
    ],
    remainder="passthrough"
)

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
print(X_train_processed.shape)
print(X_test_processed.shape)

In [ ]:
X_train_processed

In [ ]:
log_reg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

log_reg_pipeline.fit(X_train, y_train)

In [ ]:
y_pred_log = log_reg_pipeline.predict(X_test)
y_prob_log = log_reg_pipeline.predict_proba(X_test)[:, 1]

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_log))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_log))

In [ ]:
cm = confusion_matrix(y_test, y_pred_log)

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Logistic Regression Confusion Matrix")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.show()

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)

y_pred = rf_pipeline.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

In [ ]:
cm_rf= confusion_matrix(y_test, y_pred)


In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Blues")
plt.title("Random forest Confusion Matrix")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.show()

In [ ]:
from xgboost import XGBClassifier
xgb_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='logloss'
    ))
])

In [ ]:
xgb_model.fit(X_train, y_train)

In [ ]:

y_pred_xgb = xgb_model.predict(X_test)

In [ ]:
cm_rf= confusion_matrix(y_test, y_pred_xgb)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Blues")
plt.title("XGBoost  Confusion Matrix")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred_xgb))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_xgb))

In [ ]:
X_train_nn = preprocessor.fit_transform(X_train)
X_test_nn = preprocessor.transform(X_test)

In [ ]:
scaler = StandardScaler()
X_train_nn = scaler.fit_transform(X_train_nn)
X_test_nn = scaler.transform(X_test_nn)


In [ ]:
nn_model = Sequential([
    Dense(128, activation="relu", input_shape=(X_train_nn.shape[1],)),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

In [ ]:
nn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)
history = nn_model.fit(
    X_train_nn, y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=256,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
y_prob_nn = nn_model.predict(X_test_nn)
y_pred_nn = (y_prob_nn > 0.5).astype(int).ravel()

In [ ]:

print("Accuracy:", accuracy_score(y_test, y_pred_nn))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_nn))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_nn))

In [ ]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)
OneHotEncoder(handle_unknown="ignore", sparse_output=False)
OneHotEncoder(handle_unknown="ignore", sparse_output=False)

In [ ]:
nn_model = Sequential([
    Dense(128, activation="relu", input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

In [ ]:

nn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)


history = nn_model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=256,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
y_prob_nn = nn_model.predict(X_test)
y_pred_nn = (y_prob_nn > 0.5).astype(int).ravel()

In [ ]:

print("Accuracy:", accuracy_score(y_test, y_pred_nn))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_nn))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_nn))